In [1]:
import time
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from xgboost import XGBRegressor

# Load processed data from T04

data_dir = "../../Feature Engineering/data_processed"

data_train = pd.read_csv(
    os.path.join(data_dir, "data_train.csv")
)

data_test = pd.read_csv(
    os.path.join(data_dir, "data_test.csv")
)

print("Train shape:", data_train.shape)
print("Test shape:", data_test.shape)



Train shape: (9995, 135)
Test shape: (2499, 135)


In [2]:
# Separate features and target

target = "price_million"

X_train = data_train.drop(columns=[target])
y_train = data_train[target]

X_test = data_test.drop(columns=[target])
y_test = data_test[target]

# Validate feature consistency

assert list(X_train.columns) == list(X_test.columns), \
    "Train and test features do not match!"

assert not X_train.isna().any().any(), \
    "Training data contains missing values!"

assert not X_test.isna().any().any(), \
    "Testing data contains missing values!"

assert (y_train >= 0).all(), \
    "Training target contains negative prices!"

assert (y_test >= 0).all(), \
    "Testing target contains negative prices!"

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

print("\nTarget statistics:")
print(y_train.describe())
non_numeric = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

assert len(non_numeric) == 0, \
    f"Non-numeric features found: {non_numeric}"

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_test.to_numpy()).all()
assert np.isfinite(y_train.to_numpy()).all()
assert np.isfinite(y_test.to_numpy()).all()

print("Data validation passed!")

X_train: (9995, 134)
X_test: (2499, 134)

Missing values:
Train: 0
Test: 0

Target statistics:
count     9995.000000
mean       513.155361
std        390.178024
min         20.000000
25%        278.000000
50%        435.000000
75%        635.000000
max      10900.000000
Name: price_million, dtype: float64
Data validation passed!


In [3]:
# Baseline Random Forest

rf = TransformedTargetRegressor(
    regressor=RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, pred_rf))
rf_mae = mean_absolute_error(y_test, pred_rf)
rf_r2 = r2_score(y_test, pred_rf)

print("Random Forest Baseline")
print("RMSE:", round(rf_rmse, 1), "million VND")
print("MAE :", round(rf_mae, 1), "million VND")
print("R2  :", round(rf_r2, 3))

Random Forest Baseline
RMSE: 216.3 million VND
MAE : 83.1 million VND
R2  : 0.733


In [4]:
# Baseline XGBoost

xgb = TransformedTargetRegressor(
    regressor=XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        n_jobs=-1
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, pred_xgb))
xgb_mae = mean_absolute_error(y_test, pred_xgb)
xgb_r2 = r2_score(y_test, pred_xgb)

print("XGBoost Baseline")
print("RMSE:", round(xgb_rmse, 1), "million VND")
print("MAE :", round(xgb_mae, 1), "million VND")
print("R2  :", round(xgb_r2, 3))

XGBoost Baseline
RMSE: 214.0 million VND
MAE : 84.7 million VND
R2  : 0.739


In [5]:
# Hyperparameter tuning

searches = {
    "rf": (
        TransformedTargetRegressor(
            regressor=RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            ),
            func=np.log1p,
            inverse_func=np.expm1
        ),
        {
            "regressor__n_estimators": [200, 400, 600],
            "regressor__max_depth": [None, 15, 25],
            "regressor__min_samples_leaf": [1, 2, 4],
            "regressor__max_features": ["sqrt", 0.5, 1.0]
        }
    ),

    "xgb": (
        TransformedTargetRegressor(
            regressor=XGBRegressor(
                random_state=42,
                n_jobs=-1
            ),
            func=np.log1p,
            inverse_func=np.expm1
        ),
        {
            "regressor__n_estimators": [300, 600, 900],
            "regressor__learning_rate": [0.03, 0.05, 0.1],
            "regressor__max_depth": [4, 6, 8],
            "regressor__subsample": [0.7, 0.9, 1.0],
            "regressor__colsample_bytree": [0.5, 0.8, 1.0]
        }
    )
}

best = {}
tuning_results = {}

for name, (estimator, params) in searches.items():

    print(f"\nTuning {name.upper()}...")

    start_time = time.time()

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=params,
        n_iter=10,
        cv=3,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=1,
        verbose=1
    )

    search.fit(X_train, y_train)

    best[name] = search.best_estimator_

    tuning_time = (time.time() - start_time) / 60

    tuning_results[name] = {
        "Best Parameters": search.best_params_,
        "CV RMSE": -search.best_score_,
        "Time (minutes)": tuning_time
    }

    print("\nBest parameters:")
    print(search.best_params_)

    print("CV RMSE:", round(-search.best_score_, 2))
    print("Time:", round(tuning_time, 2), "minutes")


Tuning RF...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters:
{'regressor__n_estimators': 200, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 1.0, 'regressor__max_depth': None}
CV RMSE: 211.43
Time: 1.24 minutes

Tuning XGB...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best parameters:
{'regressor__subsample': 0.9, 'regressor__n_estimators': 600, 'regressor__max_depth': 8, 'regressor__learning_rate': 0.1, 'regressor__colsample_bytree': 1.0}
CV RMSE: 206.7
Time: 2.01 minutes


In [6]:
# Evaluate tuned models on the test set

tuned_predictions = {}
tuned_metrics = {}

for name, model in best.items():

    pred = model.predict(X_test)

    tuned_predictions[name] = pred

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    tuned_metrics[name] = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

    print(f"\n{name.upper()} Tuned")
    print("RMSE:", round(rmse, 1), "million VND")
    print("MAE :", round(mae, 1), "million VND")
    print("R2  :", round(r2, 3))


RF Tuned
RMSE: 216.3 million VND
MAE : 83.1 million VND
R2  : 0.733

XGB Tuned
RMSE: 206.9 million VND
MAE : 79.8 million VND
R2  : 0.755


In [7]:
# Compare baseline and tuned models

results = [
    {
        "Model": "Random Forest (baseline)",
        "RMSE": rf_rmse,
        "MAE": rf_mae,
        "R2": rf_r2
    },
    {
        "Model": "Random Forest (tuned)",
        "RMSE": tuned_metrics["rf"]["RMSE"],
        "MAE": tuned_metrics["rf"]["MAE"],
        "R2": tuned_metrics["rf"]["R2"]
    },
    {
        "Model": "XGBoost (baseline)",
        "RMSE": xgb_rmse,
        "MAE": xgb_mae,
        "R2": xgb_r2
    },
    {
        "Model": "XGBoost (tuned)",
        "RMSE": tuned_metrics["xgb"]["RMSE"],
        "MAE": tuned_metrics["xgb"]["MAE"],
        "R2": tuned_metrics["xgb"]["R2"]
    }
]

results_df = pd.DataFrame(results)

print(results_df.round(3).to_string(index=False))

                   Model    RMSE    MAE    R2
Random Forest (baseline) 216.259 83.089 0.733
   Random Forest (tuned) 216.259 83.089 0.733
      XGBoost (baseline) 213.970 84.677 0.739
         XGBoost (tuned) 206.930 79.752 0.755


In [8]:
# Select the best tuned model based on CV RMSE

best_name = min(
    tuning_results,
    key=lambda name: tuning_results[name]["CV RMSE"]
)

best_model = best[best_name]

print("Selected model:", best_name.upper())
print("CV RMSE:", round(tuning_results[best_name]["CV RMSE"], 2))
print("Best parameters:")
print(tuning_results[best_name]["Best Parameters"])

# Save selected model

joblib.dump(best_model, "best_model.pkl")

print("\nModel saved as best_model.pkl")

Selected model: XGB
CV RMSE: 206.7
Best parameters:
{'regressor__subsample': 0.9, 'regressor__n_estimators': 600, 'regressor__max_depth': 8, 'regressor__learning_rate': 0.1, 'regressor__colsample_bytree': 1.0}

Model saved as best_model.pkl
